In [1]:
from pathlib import Path

DATASET = "nyc_tlc_yellow"
YEAR = 2024
MONTH = 1

MM = f"{MONTH:02d}"
RAW_PATH = Path(f"../data/raw/yellow_tripdata_{YEAR}-{MM}.parquet")
BRONZE_PATH = f"../lakehouse/bronze/{DATASET}"

print("Run order: CONFIG -> Spark -> Load Raw -> Transform -> Write Bronze -> Validate")
print("RAW_PATH:", RAW_PATH)
print("BRONZE_PATH:", BRONZE_PATH)


Run order: CONFIG -> Spark -> Load Raw -> Transform -> Write Bronze -> Validate
RAW_PATH: ../data/raw/yellow_tripdata_2024-01.parquet
BRONZE_PATH: ../lakehouse/bronze/nyc_tlc_yellow


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYC-TLC-Lakehouse")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .getOrCreate()
)

spark


26/02/25 10:12:26 WARN Utils: Your hostname, willian-A320M-S2H resolves to a loopback address: 127.0.1.1; using 192.168.0.110 instead (on interface enp6s0)
26/02/25 10:12:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/willian/PhaifferTech/nyc-tlc-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/willian/.ivy2/cache
The jars for the packages stored in: /home/willian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b2705da5-b742-4b1e-9062-61323a5a15a5;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 377ms :: artifacts dl 19ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |  

In [3]:
df_raw = spark.read.parquet(str(RAW_PATH))

print("raw_rows:", df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)


raw_rows: 2964624
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



26/02/25 10:12:46 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [4]:
from datetime import datetime, timezone
from pyspark.sql.functions import lit

ingest_ts = datetime.fromtimestamp(RAW_PATH.stat().st_mtime, tz=timezone.utc).replace(tzinfo=None)

df_bronze = (
    df_raw
    .withColumn("ingest_ts", lit(ingest_ts).cast("timestamp"))
    .withColumn("source_file", lit(RAW_PATH.name))
)

df_bronze.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- _dataset: string (nu

In [5]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_PATH)
)

print("Bronze write completed:", BRONZE_PATH)


26/02/25 10:12:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze write completed: ../lakehouse/bronze/nyc_tlc_yellow


In [6]:
df_check = spark.read.format("delta").load(BRONZE_PATH)
bronze_rows_check = df_check.count()

print("bronze_rows_check:", bronze_rows_check)
df_check.printSchema()
df_check.show(5, truncate=False)


bronze_rows_check: 2964624
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------------+-------------------------------+--------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|_ingestion_timestamp      |_source_file                   |_dataset      |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------------+-------------

In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.events_raw")
)

print("Bronze table written: bronze.events_raw")
print("Bronze row count:", spark.table("bronze.events_raw").count())

Bronze table written: bronze.events_raw


Bronze row count: 2964624
